# NutriMatch — Food Master Cleaning Pipeline
**Role:** Data Scientist (Orang A — Nutrisi & Profil Pengguna)

**Tujuan notebook ini:**
Menghasilkan `food_master_clean.csv` yang bersih, konsisten, dan siap digunakan oleh AI Engineer untuk training dan evaluasi sistem rekomendasi NutriMatch.

**Input:** `nutrition.csv` — Indonesian Food & Drink Nutrition Dataset (1.346 baris, 7 kolom)

**Output:** `food_master_clean.csv` — satu baris per makanan, kolom terstandarisasi

---
**Pipeline tahapan:**
1. Import & Load Data
2. Initial Audit (shape, info, missing, duplicate)
3. Rename & Standardize Columns
4. Clean `food_name` (lowercase, strip, karakter valid)
5. Numeric Validation & Type Enforcement
6. Outlier Detection & Flagging
7. Macro-Calorie Cross Validation
8. Deduplication & ID Assignment
9. Final Column Selection & Export

## 1. Import Library

In [14]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2. Load Dataset & Initial Audit

In [15]:
df = pd.read_csv('nutrition.csv')

print('=' * 55)
print('INITIAL AUDIT — nutrition.csv')
print('=' * 55)
print(f'Shape         : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Duplicates    : {df.duplicated().sum()}')
print()
print('--- Column Info ---')
print(df.info())
print()
print('--- Missing Values ---')
print(df.isnull().sum())
print()
print('--- Sample Data ---')
df.head()

INITIAL AUDIT — nutrition.csv
Shape         : 1,346 rows × 7 columns
Duplicates    : 0

--- Column Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1346 entries, 0 to 1345
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            1346 non-null   int64  
 1   calories      1346 non-null   float64
 2   proteins      1346 non-null   float64
 3   fat           1346 non-null   float64
 4   carbohydrate  1346 non-null   float64
 5   name          1346 non-null   object 
 6   image         1346 non-null   object 
dtypes: float64(4), int64(1), object(2)
memory usage: 73.7+ KB
None

--- Missing Values ---
id              0
calories        0
proteins        0
fat             0
carbohydrate    0
name            0
image           0
dtype: int64

--- Sample Data ---


,id,calories,proteins,fat,carbohydrate,name,image
0,1,280.0,9.2,28.4,0.0,Abon,https://img-cdn.medkomtek.com/PbrY9X3ignQ8sVuj...
1,2,513.0,23.7,37.0,21.3,Abon haruwan,https://img-global.cpcdn.com/recipes/cbf330fbd...
2,3,0.0,0.0,0.2,0.0,Agar-agar,https://res.cloudinary.com/dk0z4ums3/image/upl...
3,4,45.0,1.1,0.4,10.8,Akar tonjong segar,https://images.tokopedia.net/img/cache/200-squ...
4,5,37.0,4.4,0.5,3.8,Aletoge segar,https://nilaigizi.com/assets/images/produk/pro...


## 3. Rename & Standardize Columns

Mengikuti schema target `food_master_clean.csv` yang disepakati dengan AI Engineer:
`food_id | food_name | calories_100g | protein_100g | fat_100g | carbohydrate_100g | source | image_url`

In [16]:
COLUMN_RENAME_MAP = {
    'name'        : 'food_name',
    'calories'    : 'calories_100g',
    'proteins'    : 'protein_100g',
    'fat'         : 'fat_100g',
    'carbohydrate': 'carbohydrate_100g',
    'image'       : 'image_url'
}

df = df.rename(columns=COLUMN_RENAME_MAP)

# Tambahkan kolom source untuk traceability
df['source'] = 'indonesian_food_nutrition_dataset'

print('Kolom setelah rename:', df.columns.tolist())

Kolom setelah rename: ['id', 'calories_100g', 'protein_100g', 'fat_100g', 'carbohydrate_100g', 'food_name', 'image_url', 'source']


## 4. Clean `food_name`

**Aturan cleaning:**
- Lowercase dan strip whitespace
- Hapus karakter selain huruf, angka, spasi, dan tanda hubung (`-`) — tanda hubung dipertahankan karena valid di nama makanan Indonesia (contoh: `es-krim`, `cap-cay`)
- Normalisasi multiple whitespace menjadi satu spasi

In [17]:
def clean_food_name(name: str) -> str:
    """
    Standardize food name:
    - Lowercase
    - Strip leading/trailing whitespace
    - Keep only alphanumeric, spaces, and hyphens
    - Collapse multiple spaces into one
    """
    name = str(name).lower().strip()
    name = re.sub(r'[^a-z0-9\s\-]', '', name)  # pertahankan '-'
    name = re.sub(r'\s+', ' ', name)            # collapse whitespace
    return name

df['food_name'] = df['food_name'].apply(clean_food_name)

# Audit: cek apakah ada food_name yang menjadi string kosong setelah cleaning
empty_names = df[df['food_name'].str.strip() == '']
print(f'Food name kosong setelah cleaning: {len(empty_names)}')

print('\nContoh food_name setelah cleaning:')
df[['food_name']].head(10)

Food name kosong setelah cleaning: 0

Contoh food_name setelah cleaning:


,food_name
0,abon
1,abon haruwan
2,agar-agar
3,akar tonjong segar
4,aletoge segar
5,alpukat segar
6,ampas kacang hijau
7,ampas tahu
8,ampas tahu kukus
9,ampas tahu mentah


## 5. Handle Missing Values & Type Enforcement

In [18]:
NUMERIC_COLS = ['calories_100g', 'protein_100g', 'fat_100g', 'carbohydrate_100g']

# Paksa ke numeric — nilai yang tidak bisa diparse menjadi NaN
for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# image_url: isi NaN dengan placeholder
df['image_url'] = df['image_url'].fillna('not_available')

# Audit missing setelah coerce
print('Missing values setelah type enforcement:')
print(df[NUMERIC_COLS].isnull().sum())

# Drop baris yang nutrisi utamanya NaN (tidak dapat diperbaiki)
rows_before = len(df)
df = df.dropna(subset=NUMERIC_COLS)
print(f'\nBaris dihapus karena NaN pada kolom numerik: {rows_before - len(df)}')

Missing values setelah type enforcement:
calories_100g        0
protein_100g         0
fat_100g             0
carbohydrate_100g    0
dtype: int64

Baris dihapus karena NaN pada kolom numerik: 0


## 6. Outlier Detection & Flagging

**Batas validasi per 100g** (berdasarkan referensi nutrisi standar):
| Kolom | Min | Max | Catatan |
|---|---|---|---|
| `calories_100g` | 0 | 900 | Lemak murni ~900 kcal/100g (batas atas teoritis) |
| `protein_100g` | 0 | 100 | Tidak mungkin > 100g per 100g bahan |
| `fat_100g` | 0 | 100 | Tidak mungkin > 100g per 100g bahan |
| `carbohydrate_100g` | 0 | 100 | Gula = ~100g/100g, ini batas valid |

**Penting:** Kalori 0 **tidak otomatis dihapus** — beberapa makanan seperti air atau agar-agar memang mendekati nol kalori. Kita flag, bukan hapus, lalu biarkan AI Engineer yang memutuskan penggunaan.

In [19]:
OUTLIER_BOUNDS = {
    'calories_100g'    : (0, 900),
    'protein_100g'     : (0, 100),
    'fat_100g'         : (0, 100),
    'carbohydrate_100g': (0, 100)
}

# Flagging baris dengan nilai di luar batas
outlier_mask = pd.Series(False, index=df.index)
for col, (lo, hi) in OUTLIER_BOUNDS.items():
    mask = (df[col] < lo) | (df[col] > hi)
    if mask.sum() > 0:
        print(f'[OUTLIER] {col}: {mask.sum()} baris di luar [{lo}, {hi}]')
        print(df.loc[mask, ['food_name', col]])
    outlier_mask |= mask

# Hapus outlier yang melampaui batas fisik
rows_before = len(df)
df = df[~outlier_mask]
print(f'\nBaris dihapus karena outlier fisik: {rows_before - len(df)}')

# --- Flag kalori nol (valid tapi perlu dicatat) ---
df['zero_calorie_flag'] = (df['calories_100g'] == 0).astype(int)
print(f'\nMakanan dengan kalori 0 (di-flag, tidak dihapus): {df["zero_calorie_flag"].sum()}')
print(df[df['zero_calorie_flag'] == 1][['food_name', 'calories_100g', 'protein_100g', 'fat_100g', 'carbohydrate_100g']])

# --- Flag nilai negatif (seharusnya tidak ada setelah batas 0, tapi tetap check) ---
neg_mask = (df[NUMERIC_COLS] < 0).any(axis=1)
print(f'\nBaris dengan nilai negatif: {neg_mask.sum()}')

[OUTLIER] calories_100g: 7 baris di luar [0, 900]
                   food_name  calories_100g
837               lemak babi          902.0
901  minyak hati hiu eulamia          902.0
902              minyak ikan          902.0
903      minyak kacang tanah          902.0
906      minyak kelapa sawit          902.0
907             minyak wijen          902.0
949            opak singkong          940.0
[OUTLIER] carbohydrate_100g: 4 baris di luar [0, 100]
                 food_name  carbohydrate_100g
653           kacang telur              124.8
750  kerupuk udang berpati              332.0
949          opak singkong              104.0
990                  pilus              647.0

Baris dihapus karena outlier fisik: 10

Makanan dengan kalori 0 (di-flag, tidak dihapus): 1
   food_name  calories_100g  protein_100g  fat_100g  carbohydrate_100g
2  agar-agar            0.0           0.0       0.2                0.0

Baris dengan nilai negatif: 0


## 7. Macro-Calorie Cross Validation

Validasi silang: estimasi kalori dari makro (`protein×4 + fat×9 + carb×4`) dibandingkan dengan nilai `calories_100g` yang tercatat.

Selisih besar (> 50 kcal) bisa menunjukkan:
- Data entry error
- Kandungan alkohol (7 kcal/g) atau serat yang belum dihitung
- Dataset tidak akurat

In [20]:
# Estimasi kalori dari makronutrien
df['calorie_estimate_from_macro'] = (
    df['protein_100g'] * 4 +
    df['fat_100g']     * 9 +
    df['carbohydrate_100g'] * 4
).round(2)

df['calorie_macro_diff'] = (df['calories_100g'] - df['calorie_estimate_from_macro']).round(2)

DIFF_THRESHOLD = 50  # kcal
df['calorie_inconsistent_flag'] = (df['calorie_macro_diff'].abs() > DIFF_THRESHOLD).astype(int)

n_inconsistent = df['calorie_inconsistent_flag'].sum()
print(f'Makanan dengan selisih kalori > {DIFF_THRESHOLD} kcal: {n_inconsistent} baris')
print(f'({n_inconsistent/len(df)*100:.1f}% dari total data)')

print('\nContoh baris tidak konsisten:')
df[df['calorie_inconsistent_flag'] == 1][
    ['food_name', 'calories_100g', 'calorie_estimate_from_macro', 'calorie_macro_diff']
].head(10)

Makanan dengan selisih kalori > 50 kcal: 58 baris
(4.3% dari total data)

Contoh baris tidak konsisten:


,food_name,calories_100g,calorie_estimate_from_macro,calorie_macro_diff
53,babi hutan masak rica masakan,491.0,245.9,245.1
132,beras siger,344.0,119.4,224.6
141,biji jambu mete,562.0,625.6,-63.6
174,bubur,60.0,596.0,-536.0
176,bubur tinotuan manado,156.0,73.4,82.6
186,bungkil kacang tanah,336.0,388.6,-52.6
204,cassavastick,460.0,240.3,219.7
212,coklat manis batang,472.0,527.0,-55.0
213,coklat pahit batang,504.0,614.9,-110.9
214,coklat susu batang,381.0,565.4,-184.4


## 8. Deduplication & Stable Food ID Assignment

In [21]:
rows_before = len(df)
df = df.drop_duplicates(subset='food_name', keep='first')
print(f'Duplikat berdasarkan food_name dihapus: {rows_before - len(df)} baris')

# Reset index dan assign food_id yang stabil
df = df.reset_index(drop=True)
df['food_id'] = df.index.map(lambda i: f'FD{i+1:04d}')

print(f'Total baris final: {len(df):,}')
print(f'Contoh food_id: {df["food_id"].head(5).tolist()}')

Duplikat berdasarkan food_name dihapus: 4 baris
Total baris final: 1,332
Contoh food_id: ['FD0001', 'FD0002', 'FD0003', 'FD0004', 'FD0005']


## 9. Final Column Selection & Export

Kolom final sesuai schema yang disepakati dengan AI Engineer + kolom flag audit untuk transparansi.

In [22]:
# Kolom wajib (schema handover ke AI Engineer)
CORE_COLUMNS = [
    'food_id',
    'food_name',
    'calories_100g',
    'protein_100g',
    'fat_100g',
    'carbohydrate_100g',
    'source',
    'image_url'
]

# Kolom audit (tambahan untuk transparansi — AI Engineer dapat menggunakan untuk filtering)
AUDIT_COLUMNS = [
    'zero_calorie_flag',
    'calorie_inconsistent_flag',
    'calorie_macro_diff'
]

food_master_clean = df[CORE_COLUMNS + AUDIT_COLUMNS].copy()

print('=' * 55)
print('FOOD MASTER CLEAN — SUMMARY')
print('=' * 55)
print(f'Total rows         : {len(food_master_clean):,}')
print(f'Total columns      : {len(food_master_clean.columns)}')
print(f'Missing values     : {food_master_clean.isnull().sum().sum()}')
print(f'Zero calorie foods : {food_master_clean["zero_calorie_flag"].sum()}')
print(f'Inconsistent cal   : {food_master_clean["calorie_inconsistent_flag"].sum()}')
print()
print('--- Distribusi Statistik Nutrisi ---')
print(food_master_clean[['calories_100g', 'protein_100g', 'fat_100g', 'carbohydrate_100g']].describe().round(2))
print()
food_master_clean.head()

FOOD MASTER CLEAN — SUMMARY
Total rows         : 1,332
Total columns      : 11
Missing values     : 0
Zero calorie foods : 1
Inconsistent cal   : 58

--- Distribusi Statistik Nutrisi ---
       calories_100g  protein_100g  fat_100g  carbohydrate_100g
count        1332.00       1332.00   1332.00            1332.00
mean          198.74         10.01      7.09              24.65
std           154.60         11.85     12.15              25.83
min             0.00          0.00      0.00               0.00
25%            74.00          1.90      0.50               4.68
50%           144.50          5.00      2.00              13.30
75%           331.00         14.92      7.93              37.35
max           884.00         83.00    100.00              95.30



,food_id,food_name,calories_100g,protein_100g,fat_100g,carbohydrate_100g,source,image_url,zero_calorie_flag,calorie_inconsistent_flag,calorie_macro_diff
0,FD0001,abon,280.0,9.2,28.4,0.0,indonesian_food_nutrition_dataset,https://img-cdn.medkomtek.com/PbrY9X3ignQ8sVuj...,0,0,-12.4
1,FD0002,abon haruwan,513.0,23.7,37.0,21.3,indonesian_food_nutrition_dataset,https://img-global.cpcdn.com/recipes/cbf330fbd...,0,0,0.0
2,FD0003,agar-agar,0.0,0.0,0.2,0.0,indonesian_food_nutrition_dataset,https://res.cloudinary.com/dk0z4ums3/image/upl...,1,0,-1.8
3,FD0004,akar tonjong segar,45.0,1.1,0.4,10.8,indonesian_food_nutrition_dataset,https://images.tokopedia.net/img/cache/200-squ...,0,0,-6.2
4,FD0005,aletoge segar,37.0,4.4,0.5,3.8,indonesian_food_nutrition_dataset,https://nilaigizi.com/assets/images/produk/pro...,0,0,-0.3


In [23]:
OUTPUT_PATH = 'food_master_clean.csv'
food_master_clean.to_csv(OUTPUT_PATH, index=False)
print(f'Berhasil disimpan: {OUTPUT_PATH}')
print(f'   Shape: {food_master_clean.shape}')

Berhasil disimpan: food_master_clean.csv
   Shape: (1332, 11)
